# 1b · Does percentile sensitivity vary by development context?

**Research question (RQ 1b):** *Does IQB parameter sensitivity vary by development context, particularly in countries with uneven infrastructure that likely exhibit wider measurement distributions? Their IQB scores and rankings may shift more dramatically across percentile thresholds than countries with uniformly developed networks.*

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau

import iqb_tools as T

T.set_style()
rng = np.random.default_rng(4)  # for the bootstrap intervals
cfg = T.config_table()
ctx = T.read_table("country_context").set_index("country_code")  # written by 0-setup

# 2025 pass rates per country: the test-weighted average of the 12 months
df25 = T.load_level("country", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS"))
tests = T.distinct_tests(cfg)
C = T.pool_pass_rates(df25, T.pass_rates(df25, tests), "country_code", tests)
S = T.iqb_scores(C, cfg, T.Q_GRID)  # IQB score of each country at P1..P99

# Sample: at least the minimum number of tests in 2025 (1a) and a World Bank income group
d = ctx[(T.binding_tests(ctx) >= T.MIN_TESTS) & ctx.income_group.notna()].copy()
d["income"] = pd.Categorical(d.income_group, T.INCOME_ORDER, ordered=True)
print(f"{len(d)} countries:", d.income.value_counts(sort=False).to_dict())

The printout gives the sample we use for the rankings: the 201 countries with at least 1,153 tests in 2025 (the minimum from 1a) and a World Bank income group, by income group. The same cell computes `S`, every country's IQB score at each percentile from P1 to P99, from its 2025 pass rates.

## Which thresholds impact the score the most? 

In [ ]:
# How each country's 2025 P50 score would be displayed: the minimum number of tests (1a),
# then the confidence that the true score is within ±0.05 of the one shown
rows = []
for cc, row in C[C.n_down >= T.FLOOR_TESTS].iterrows():
    u = T.score_uncertainty(row, row.n_down, (50,), key=f"{cc}|2025").loc[50]
    rows.append(
        {
            "country_code": cc,
            "n_2025": row.n_down,
            "iqb_p50": u.score,
            "range_p50_lo": u.lo,
            "range_p50_hi": u.hi,
            "conf_p50_2025": u.conf,
            "closest_test_p50": u.closest_test,
            "closest_pass_rate": u.closest_pass_rate,
            "display_2025": T.display_decision(min(row.n_down, row.n_up), u.conf),
        }
    )
R = pd.DataFrame(rows).set_index("country_code").join(ctx[["country", "income_group"]])
R.display_2025.value_counts()

The table counts how each country's 2025 P50 score would be displayed under the rules of 1a. It covers the 234 countries with at least 100 download tests in 2025, not only the 201 above, so places without an income group are included:

| Decision | Meaning |
|---|---|
| *show the score* | At least 1,153 tests in both directions, and at least 95% confidence that the true score is within ±0.05 of the one shown. |
| *show a range* | Enough tests, but less than 95% confidence, so the 95% interval of plausible scores is shown instead of one number. |
| *pool more data* | Between 100 and 1,153 tests: too few for a score without pooling more months. |

With a whole year of tests, 191 countries can show a single number, 20 need a range and 23 need more data.

In [ ]:
unsure = R[(R.display_2025 == "show a range")].sort_values("n_2025", ascending=False)
print(f"{len(unsure)} countries need a range in 2025. The pass rate closest to 50% decides it:")
print(unsure.closest_test_p50.value_counts().to_string())
unsure[["country", "income_group", "n_2025", "iqb_p50", "range_p50_lo", "range_p50_hi", "conf_p50_2025",
        "closest_test_p50", "closest_pass_rate"]].head(12).round(3)

The printed list takes the 20 countries that need a range and counts the requirement whose pass rate is closest to 50%, which is the one that makes their score uncertain. The table shows the 12 of them with the most tests:

| Column | Meaning |
|---|---|
| *n_2025* | Download tests in 2025. |
| *iqb_p50* | The P50 score computed from the 2025 pass rates. |
| *range_p50_lo*, *range_p50_hi* | The 95% interval of plausible true scores, shown instead of the score. |
| *conf_p50_2025* | The confidence that the true score is within ±0.05 of *iqb_p50* (0.507 means 50.7%). Below 0.95, a range is shown. |
| *closest_test_p50* | The requirement whose pass rate is closest to 50%. |
| *closest_pass_rate* | That pass rate. |

In every row the deciding pass rate is between 48.8% and 50.9%, whatever the number of tests. Sri Lanka has more than 800,000 tests and the lowest confidence in the table (50.7%), because its pass rate for download above 10 Mbit/s is 50.0%. In most rows the score is one end of its range: the deciding requirement either passes or fails, so if the estimate is wrong, the true score lies on one side only. Botswana's and DR Congo's ranges extend on both sides because each has two pass rates within about 0.7 points of 50%.

## Do rankings shift more in some groups than in others?

We measure how much rankings shift between two percentiles with Kendall's τ, computed within each group: τ-b with a 95% bootstrap interval from resampling countries within the group. A τ of 1 means both percentiles rank the group's countries in the same order.

In [ ]:
PAIRS = [(25, 50), (50, 75), (75, 95), (50, 95), (25, 75)]


def tau_ci(a, b, n_boot=1000):
    a, b = np.asarray(a), np.asarray(b)
    t = kendalltau(a, b).statistic
    bs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(a), len(a))
        bs.append(kendalltau(a[i], b[i]).statistic)
    lo, hi = np.nanpercentile(bs, [2.5, 97.5])
    return t, lo, hi


def tau_table(groups):
    out = []
    for name, idx in groups.items():
        for p, q in PAIRS:
            t, lo, hi = tau_ci(S.loc[idx, p], S.loc[idx, q])
            out.append((name, f"P{p} vs P{q}", t, lo, hi, len(idx)))
    return pd.DataFrame(out, columns=["group", "pair", "tau", "lo", "hi", "n"])


groups_inc = {
    "All": d.index,
    **{T.INCOME_SHORT[i]: d.index[d.income == i] for i in T.INCOME_ORDER},
}
groups_hhi = {
    h: d.index[d.hhi_tercile == h] for h in ["Low HHI", "Mid HHI", "High HHI"]
}
tau_inc, tau_hhi = tau_table(groups_inc), tau_table(groups_hhi)
tau_all = pd.concat([tau_inc, tau_hhi])
tau_all.pivot(index="group", columns="pair", values="tau").loc[
    list(groups_inc) + list(groups_hhi), [f"P{p} vs P{q}" for p, q in PAIRS]
].round(2)


Each value is Kendall's τ between the rankings of the same countries at two percentiles (columns), computed within the group in the row. Roughly, and ignoring ties, τ is the share of pairs of countries that both percentiles put in the same order minus the share they put in opposite orders: 1 means identical rankings and 0 means no relation, and 0.60 means the two percentiles agree on about 80% of pairs and disagree on about 20%. The rows are all 201 countries, the four income groups, and the three terciles of ISP market concentration (HHI) from 0-setup.

For all countries together, neighboring percentiles agree more (0.67–0.73 for P25–P50, P50–P75 and P75–P95) than distant ones (0.60 for P50–P95, 0.63 for P25–P75). Within a group, τ is usually lower than for all countries, because the countries in a group are more alike and small score differences decide their order. The lowest values are for lower-middle-income countries between P25 and P50 (0.37) and between P25 and P75 (0.38).

The groups are small, so these values are uncertain: the 95% bootstrap intervals saved in `kendall_tau.csv` are typically 0.25–0.4 wide for the income groups (widest for the 22 low-income countries) and about 0.2 for the HHI terciles. For P50 vs P95, the interval runs from 0.50 to 0.82 for low-income countries and from 0.22 to 0.57 for upper-middle-income ones. The two overlap, so even the largest gap between income groups in that column is not clearly real.

## Save

In [ ]:
R.to_csv(T.TAB / "display_decisions_2025.csv")
tau_all.to_csv(T.TAB / "kendall_tau.csv", index=False)
print("saved display_decisions_2025.csv", R.shape, "and kendall_tau.csv", tau_all.shape, "to", T.TAB)

## Summary

**With a year of tests, most countries' P50 score can be displayed as a single number.** Of the 234 countries with at least 100 tests, 191 show a score, 20 need a range and 23 need more data. The 20 ranges are not a matter of too few tests. Every one of them has a pass rate within about two percentage points of 50% (half a point in the typical case).

**The requirement that impacts the score is most often packet loss** (14 of the 20: 10 for loss below 1% and 4 for loss below 0.5%), followed by download (3) and upload (3). Some of these countries have large samples: Sri Lanka has over 800,000 tests, and its P50 score is still only 51% likely to be within ±0.05.

**Rankings shift with the percentile.** Within each income group, the P50 and P95 rankings agree only moderately, with Kendall's τ between 0.41 for upper-middle-income and 0.67 for low-income countries, and 0.60 for all countries together. Across every pair of percentiles, τ ranges from 0.37 to 0.79. By ISP concentration, τ between P50 and P95 is 0.59, 0.53 and 0.67 in the low, middle and high HHI terciles. The table shows point estimates; the bootstrap intervals are saved with it in `kendall_tau.csv`, and they should be checked before reading a difference between groups as real.